# Artificial Neural Network

### Importing the libraries

In [0]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
tf.__version__

'2.2.0'

## Part 1 - Data Preprocessing

### Importing the dataset

In [0]:
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

In [4]:
print(X)

[[619 'France' 'Female' ... 1 1 101348.88]
 [608 'Spain' 'Female' ... 0 1 112542.58]
 [502 'France' 'Female' ... 1 0 113931.57]
 ...
 [709 'France' 'Female' ... 0 1 42085.58]
 [772 'Germany' 'Male' ... 1 0 92888.52]
 [792 'France' 'Female' ... 1 0 38190.78]]


In [5]:
print(y)

[1 0 1 ... 1 1 0]


### Encoding categorical data

Label Encoding the "Gender" column

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [7]:
print(X)

[[619 'France' 0 ... 1 1 101348.88]
 [608 'Spain' 0 ... 0 1 112542.58]
 [502 'France' 0 ... 1 0 113931.57]
 ...
 [709 'France' 0 ... 0 1 42085.58]
 [772 'Germany' 1 ... 1 0 92888.52]
 [792 'France' 0 ... 1 0 38190.78]]


One Hot Encoding the "Geography" column

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [9]:
print(X)

[[1.0 0.0 0.0 ... 1 1 101348.88]
 [0.0 0.0 1.0 ... 0 1 112542.58]
 [1.0 0.0 0.0 ... 1 0 113931.57]
 ...
 [1.0 0.0 0.0 ... 0 1 42085.58]
 [0.0 1.0 0.0 ... 1 0 92888.52]
 [1.0 0.0 0.0 ... 1 0 38190.78]]


### Splitting the dataset into the Training set and Test set

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Feature Scaling

In [0]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

### Initializing the ANN

In [0]:
ann = tf.keras.models.Sequential()

### Adding the input layer and the first hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the second hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the output layer

In [0]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the ANN

### Compiling the ANN

In [0]:
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the ANN on the Training set

In [17]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
250/250 [==============================] - 0s 1ms/step - loss: 0.8037 - accuracy: 0.5185
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 0.5291 - accuracy: 0.7901
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4888 - accuracy: 0.7952
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4668 - accuracy: 0.7979
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4478 - accuracy: 0.7994
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4302 - accuracy: 0.8049
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4123 - accuracy: 0.8119
Epoch 8/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3947 - accuracy: 0.8238
Epoch 9/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3807 - accuracy: 0.8355
Epoch 10/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3720 - accura

## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

**Homework**

Use our ANN model to predict if the customer with the following informations will leave the bank: 

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer?

**Solution**

In [18]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

[[False]]


Therefore, our ANN model predicts that this customer stays in the bank!

**Important note 1:** Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

**Important note 2:** Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

### Predicting the Test set results

In [19]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


### Making the Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1516   79]
 [ 200  205]]


0.8605

# Study Notes: Preprocessing Data for the Bank-Churn ANN

## 1. Objective and workflow

The model predicts whether a bank customer leaves: `Exited = 1` means the customer left, while `Exited = 0` means the customer stayed. This is a **binary-classification** problem. The preprocessing workflow is:

1. Load and inspect the data.
2. Select useful predictors and the target.
3. Split the data into training and test sets.
4. Fit categorical encoders and scalers on the training data only.
5. Apply those fitted transformations to both sets.
6. Feed the resulting numerical arrays into the ANN.

> The phrase *artificial brain* is motivational shorthand. The implementation is a numerical classifier inspired only loosely by biological neural systems.

## 2. Imports and environment

```python
import numpy as np
import pandas as pd
import tensorflow as tf
```

- NumPy supplies numerical arrays.
- pandas loads and inspects tabular data.
- TensorFlow/Keras builds and trains the neural network.
- Matplotlib is unnecessary unless the notebook later creates plots.

`tf.__version__` reports the installed TensorFlow version. The saved output in this historical notebook is `2.2.0`; a current Colab runtime may provide a different release. For reproducible projects, record or pin the complete package versions rather than checking only that TensorFlow's major version is 2.

## 3. Load and inspect the dataset

```python
dataset = pd.read_csv('Churn_Modelling.csv')
```

Before modeling, inspect more than the first few rows:

```python
dataset.head()
dataset.shape
dataset.info()
dataset.isna().sum()
dataset.duplicated().sum()
dataset['Exited'].value_counts(normalize=True)
```

These checks reveal schema problems, missing values, duplicates, and target imbalance. Never assume a production dataset is clean merely because a course copy is clean.

## 4. Select predictors and target

The notebook uses positional indexing:

```python
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values
```

This excludes `RowNumber`, `CustomerId`, and `Surname`, and selects `Exited` as the target. Named columns are safer because column reordering will not silently change the model inputs:

```python
excluded = ['RowNumber', 'CustomerId', 'Surname', 'Exited']
X = dataset.drop(columns=excluded)
y = dataset['Exited']
```

Keeping `X` as a DataFrame also preserves column names for later preprocessing and debugging.

Identifiers are usually excluded because they do not represent stable customer behavior. Do not justify exclusion only by claiming that a field has *no impact*: identifiers and surnames can accidentally proxy for time, location, ethnicity, or data-collection processes. Evaluate every feature for predictive validity, leakage, availability at inference time, privacy, stability, and fairness.

## 5. Encode categorical variables

The categorical predictors are `Geography` and `Gender`. A neural network expects numerical input, so strings must be encoded.

### Gender in the course code

```python
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])
```

For this dataset, alphabetical class ordering normally produces `Female -> 0` and `Male -> 1`. This assignment is **deterministic, not random**; inspect it with `le.classes_`. In current scikit-learn, `LabelEncoder` is intended for target labels (`y`), not input columns (`X`). For predictors, prefer `OneHotEncoder`, `OrdinalEncoder`, or an explicit validated mapping. An explicit map must still define how missing or previously unseen values are handled.

### Geography with one-hot encoding

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), [1])],
    remainder='passthrough'
)
X = np.array(ct.fit_transform(X))
```

One-hot encoding creates one indicator column per observed category without imposing a false ranking such as France < Germany < Spain. `ColumnTransformer` places transformed columns first and then passes through the remaining columns. Inspect the learned categories and output names rather than hard-coding their order:

```python
ct.named_transformers_['encoder'].categories_
ct.get_feature_names_out()
```

For inference on new customers, a robust encoder commonly uses:

```python
OneHotEncoder(handle_unknown='ignore', sparse_output=False)
```

With `handle_unknown='ignore'`, an unseen category does not crash transformation; it becomes all zeros for that feature's encoded columns. That behavior should still be monitored as possible data drift. Older scikit-learn versions use `sparse=False` instead of `sparse_output=False`.

## 6. Split before fitting learned preprocessing

A safer general workflow splits the raw predictors before learning category vocabularies or scaling statistics:

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0, stratify=y
)
```

- `test_size=0.20` reserves 20% of rows for final evaluation.
- `random_state=0` makes the split reproducible.
- `stratify=y` approximately preserves the churn ratio in both sets.

The notebook one-hot encodes before splitting. That is a convenient teaching shortcut, but the general rule is: **split first; fit every data-dependent transformation on the training set only**. This prevents test-set information from influencing the training pipeline.

## 7. Feature scaling

The course standardizes every transformed feature:

```python
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
```

For each feature, `StandardScaler` applies approximately $z = (x - \mu) / \sigma$, using the training-set mean and standard deviation. Fitting only on `X_train` avoids leakage; `X_test` must use the same learned statistics.

Scaling is generally very helpful for dense neural networks because similarly scaled inputs make gradient-based optimization more stable and efficient. It is not an absolute mathematical requirement in every architecture. Binary and one-hot features are already bounded, so many modern pipelines scale continuous columns and pass binary indicators through unchanged. Scaling every column, as this notebook does, is also workable.

## 8. Recommended named-column preprocessor

This version is easier to audit, fits preprocessing only on the training data, supports unseen categories, and preserves a reproducible transformation:

```python
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = dataset.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = dataset['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0, stratify=y
)

numeric_features = [
    'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'EstimatedSalary'
]
categorical_features = ['Geography', 'Gender']
binary_features = ['HasCrCard', 'IsActiveMember']

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False
        ), categorical_features),
        ('binary', 'passthrough', binary_features),
    ],
    remainder='drop'
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
```

With all three geography categories and both gender categories represented in training, this configuration produces 13 inputs: 6 scaled numerical features, 5 one-hot columns, and 2 binary features. The exact width depends on categories learned from the training data.

If missing values are possible, place a `SimpleImputer(strategy='median')` before the numeric scaler and a `SimpleImputer(strategy='most_frequent')` before the categorical encoder, using scikit-learn `Pipeline` objects. Choose imputation rules from domain knowledge and validate them rather than applying them blindly.

## 9. Validation checks

Before building the ANN, verify:

```python
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()
preprocessor.get_feature_names_out()
```

Also confirm that the encoded feature order used during training is exactly the order used at inference. Never manually assemble a production input vector by guessing the dummy-column positions.

## 10. Fairness and deployment considerations

`Gender` and `Geography` can be sensitive attributes or proxies for protected characteristics. Their use requires legal, ethical, and domain review. Removing them does not automatically remove bias because other fields may be proxies; evaluate performance and error rates across relevant groups.

Save the fitted preprocessor together with the trained ANN. The production path should be:

`raw customer row -> schema validation -> preprocessor.transform -> ANN -> churn probability -> decision threshold`

Monitor missingness, unseen categories, feature distributions, class balance, calibration, and subgroup performance. The statement that preprocessing is *70% of data-science work* is an informal reminder that data work is substantial, not a universal measured rule.

## 11. Common pitfalls

- Fitting encoders or scalers on the complete dataset.
- Calling `fit_transform` on the test set instead of `transform`.
- Relying on fragile positional column indexes.
- Treating nominal categories as ordered integers.
- Assuming category-to-integer assignments are random.
- Failing on unseen categories in production.
- Losing the transformed feature names and order.
- Keeping identifiers that encourage memorization or leakage.
- Scaling before the train/test split.
- Evaluating only accuracy on an imbalanced target.

## 12. Key takeaways

- Select predictors by meaning and name, not only by position.
- Split before fitting any learned preprocessing step.
- One-hot encode nominal predictors and handle unseen values deliberately.
- Fit scaling statistics on training data and reuse them unchanged.
- Keep preprocessing and model artifacts together for inference.
- Treat leakage, fairness, drift, and reproducibility as part of model quality.

## 13. Review questions

1. Why are `RowNumber`, `CustomerId`, and `Surname` excluded?
2. What could go wrong if those fields are retained?
3. Why is one-hot encoding appropriate for `Geography`?
4. Is `Female -> 0` and `Male -> 1` chosen randomly?
5. Why is `LabelEncoder` not the preferred current tool for predictor columns?
6. Why should the data be split before encoders and scalers are fitted?
7. What does `stratify=y` accomplish?
8. Why is the scaler fitted on `X_train` but only applied to `X_test`?
9. Must one-hot and binary columns always be standardized?
10. How should a deployed model handle a new geography?
11. Why must the preprocessor be saved with the neural network?
12. Which metrics would you examine in addition to accuracy for churn prediction?

## Further reading

- [scikit-learn: preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)
- [scikit-learn: composing estimators](https://scikit-learn.org/stable/modules/compose.html)
- [scikit-learn: common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html)
- [TensorFlow: structured-data classification tutorial](https://www.tensorflow.org/tutorials/structured_data/preprocessing_layers)

# Study Notes: Building the Artificial Neural Network

## 1. Architecture at a glance

This section constructs a feed-forward, fully connected network for binary churn classification:

`preprocessed features -> Dense(6, ReLU) -> Dense(6, ReLU) -> Dense(1, sigmoid)`

The four construction steps are:

1. Initialize a Keras `Sequential` model.
2. Define the input shape and add the first hidden layer.
3. Add a second hidden layer.
4. Add the binary-classification output layer.

At this point the network is only **defined**. It still must be compiled and trained before it can make useful predictions.

## 2. The `Sequential` model

The notebook initializes the model with:

```python
ann = tf.keras.models.Sequential()
```

A `Sequential` model represents a linear stack: each layer receives the preceding layer's output. It is a good fit when the model has one straightforward path from input to output.

For architectures with multiple inputs or outputs, shared layers, residual connections, or branches, use the Keras Functional API or subclass `tf.keras.Model`. Describing every non-sequential architecture as a Boltzmann machine is misleading; Boltzmann machines are a distinct family of probabilistic models.

## 3. Dense layers

A dense layer connects every input value to every neuron in that layer. Conceptually, it computes:

$$\mathbf{h} = \phi(\mathbf{xW} + \mathbf{b})$$

where $\mathbf{x}$ is the input, $\mathbf{W}$ contains trainable weights, $\mathbf{b}$ contains trainable biases, and $\phi$ is the activation function.

The TensorFlow/Keras class is `tf.keras.layers.Dense`. PyTorch does **not** use the same class name; its comparable fully connected layer is `torch.nn.Linear`.

## 4. Define the input explicitly

The original notebook lets Keras infer the input width when training begins. That works, but an explicit input is clearer and lets `model.summary()` display the complete model immediately:

```python
ann = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
```

`X_train.shape[1]` is the number of features produced by preprocessing. In the notebook's original preprocessing path, geography expands to three columns while gender remains one encoded column, producing 12 inputs. If the preprocessing strategy changes—for example, if gender is also one-hot encoded—the input width can change. Deriving it from `X_train` prevents a hard-coded mismatch.

An equivalent incremental construction is:

```python
ann = tf.keras.Sequential()
ann.add(tf.keras.Input(shape=(X_train.shape[1],)))
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
```

Both styles describe the same architecture. The list form is often easier to scan, while repeated `add()` calls can be convenient when constructing a model conditionally.

## 5. Hidden layers and ReLU

```python
tf.keras.layers.Dense(units=6, activation='relu')
```

- `units=6` creates six neurons and therefore six output values from the layer.
- `activation='relu'` applies $\max(0, z)$ elementwise.
- The layer's weights and biases are learned during training.

ReLU is a strong, common default for hidden layers because it is simple and generally supports effective gradient-based training. It is not a rule that hidden layers **must** use ReLU; alternatives such as GELU, ELU, tanh, and others may be suitable depending on the architecture and task.

Likewise, `units=6` is a starting design choice—not a value derived by a universal formula. Select layer width, depth, activation, regularization, and learning rate using domain constraints and validation performance. Do not repeatedly tune them against the final test set, because that leaks test information into model selection.

## 6. What makes a network 'deep'?

Adding a second hidden layer allows the model to compose transformations: the second layer learns from features created by the first. This may represent more complex relationships than a single linear transformation.

There is no universally enforced cutoff at which a network becomes *deep*. The term generally refers to having multiple learned layers, but merely adding a second hidden layer does not guarantee better performance. A deeper model can be unnecessary for a small tabular dataset and may increase overfitting, training time, and tuning complexity. Compare candidate architectures on validation data.

## 7. The binary output layer

```python
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
```

One output neuron is sufficient because the target has two complementary outcomes. The sigmoid maps the output logit to a value between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This value is trained as the model's estimate of $P(\text{Exited}=1 \mid \mathbf{x})$. A threshold such as 0.5 can later convert the score into a class label, although the best operational threshold depends on the relative cost of false positives and false negatives.

A sigmoid score is not automatically a well-calibrated real-world probability. Check calibration on held-out data when probability quality matters. The later compilation step should pair this output with binary cross-entropy.

### Output designs by task

| Task | Output layer | Typical loss |
|---|---|---|
| Binary classification | `Dense(1, activation='sigmoid')` | Binary cross-entropy |
| Mutually exclusive $K$-class classification | `Dense(K, activation='softmax')` | Sparse or categorical cross-entropy |
| Multi-label classification | `Dense(K, activation='sigmoid')` | Binary cross-entropy |
| Regression | Often `Dense(1)` with no activation | A regression loss such as MSE or MAE |

For multiclass classification, labels do not always need manual one-hot encoding: integer labels can be used with sparse categorical cross-entropy, while one-hot labels pair with categorical cross-entropy.

## 8. Inspect the model

```python
ann.summary()
```

With 12 input features and the 6-6-1 architecture, the trainable parameter count is:

- First dense layer: $(12 \times 6) + 6 = 78$
- Second dense layer: $(6 \times 6) + 6 = 42$
- Output layer: $(6 \times 1) + 1 = 7$
- Total: **127 trainable parameters**

Each formula is `input connections x neurons + one bias per neuron`. If the preprocessor produces a different number of input columns, the first layer's parameter count changes accordingly.

## 9. Architecture choices to validate later

- Number of hidden layers and units per layer.
- Activation functions.
- Weight regularization or dropout if overfitting occurs.
- Optimizer and learning rate.
- Batch size and number of epochs.
- Decision threshold and probability calibration.

Change one well-motivated factor at a time or use a reproducible search strategy. Compare models using a validation set or cross-validation, then evaluate the chosen design once on the untouched test set.

## 10. Common pitfalls

- Hard-coding an input width that no longer matches preprocessing output.
- Forgetting that `Dense` includes trainable biases by default.
- Assuming more layers or neurons always improve accuracy.
- Calling ReLU mandatory rather than a common default.
- Using sigmoid for a mutually exclusive multiclass output.
- Treating sigmoid scores as perfectly calibrated probabilities without checking.
- Selecting architecture hyperparameters using the test set.
- Confusing model construction with compilation or training.

## 11. Key takeaways

- `Sequential` is appropriate for a simple linear stack of layers.
- `Dense` means every input is connected to every neuron in that layer.
- Derive the input shape from the transformed training data.
- ReLU is a practical hidden-layer default, not a requirement.
- One sigmoid output matches a binary target.
- Layer widths and depth are hyperparameters to validate, not learnable weights.
- Use `summary()` to verify shapes and parameter counts before training.

## 12. Review questions

1. When is a `Sequential` model inappropriate?
2. What computation does a dense layer perform?
3. Why is an explicit `Input` layer useful?
4. Why should the input width come from `X_train.shape[1]`?
5. What does `units=6` mean?
6. Why is ReLU commonly used in hidden layers?
7. Is a second hidden layer guaranteed to improve the model?
8. Why does binary classification need only one output neuron?
9. Which output and loss would you use for five mutually exclusive classes?
10. How many parameters are in a dense layer with 12 inputs and 6 neurons?
11. Why should architecture choices be evaluated on validation rather than test data?
12. What remains to be done after constructing the model?

## Further reading

- [TensorFlow: the Sequential model](https://www.tensorflow.org/guide/keras/sequential_model)
- [TensorFlow API: Dense layer](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
- [TensorFlow: Functional API](https://www.tensorflow.org/guide/keras/functional_api)
- [TensorFlow: classification on imbalanced structured data](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data)

# Study Notes: Compiling and Training the ANN

## 1. From architecture to learning

Building the network defines its layers, but it does not train the model. The remaining workflow has two stages:

1. **Compile** the ANN by choosing an optimizer, loss function, and evaluation metrics.
2. **Fit** the ANN to the training data over a chosen number of epochs and with a chosen batch size.

The notebook uses:

```python
ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history = ann.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,
)
```

`compile()` configures how the model will learn; `fit()` performs the learning.

## 2. What `compile()` configures

### Optimizer: Adam

The optimizer updates the trainable weights and biases to reduce the loss. Adam is a widely used gradient-based optimizer that adapts the effective learning rate for each parameter using estimates of the gradients' first and second moments.

```python
optimizer='adam'
```

The correct name is **Adam**, not "Atom." Passing the string asks Keras to construct Adam with its default settings. For explicit control, use an optimizer object:

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
```

Adam is a useful starting point, but it is not guaranteed to be best for every dataset. The optimizer and learning rate are hyperparameters that should be assessed using validation data.

### Loss: binary cross-entropy

The loss function supplies the numerical objective that the optimizer minimizes. Because this network has one sigmoid output and a binary target, binary cross-entropy is the appropriate pairing:

$$L = -(1/N) sum_{i=1}^{N} [y_i log(p_i) + (1-y_i) log(1-p_i)]$$

Here, $y_i$ is the true label and $p_i$ is the predicted score for class 1. Confident incorrect predictions receive a large penalty.

The output/loss combination must be internally consistent:

| Task | Typical output | Typical loss |
|---|---|---|
| Binary classification | One sigmoid unit | `binary_crossentropy` |
| Mutually exclusive classes with integer labels | $K$ softmax units | `sparse_categorical_crossentropy` |
| Mutually exclusive classes with one-hot labels | $K$ softmax units | `categorical_crossentropy` |
| Multi-label classification | $K$ independent sigmoid units | `binary_crossentropy` |
| Regression | Usually a linear output | A regression loss such as MSE or MAE |

If the output layer returns raw logits instead of probabilities, configure the loss with `from_logits=True` and do not also apply sigmoid or softmax in the output layer.

### Metric: accuracy

```python
metrics=['accuracy']
```

Metrics are reported for monitoring and evaluation; they are not the objective optimized unless they are also used as the loss. The `metrics` argument is a list because several metrics may be tracked at once.

Accuracy is easy to interpret, but churn datasets may be imbalanced. Also examine a confusion matrix and metrics such as precision, recall, F1, ROC-AUC, or PR-AUC when the costs of missed churners and false alarms differ.

## 3. What `fit()` does

```python
history = ann.fit(X_train, y_train, batch_size=32, epochs=100)
```

During training, Keras repeatedly:

1. Runs a batch through the network to produce predictions.
2. Computes the loss by comparing predictions with the true labels.
3. Uses backpropagation to calculate gradients.
4. Lets Adam update the parameters.
5. Continues until all batches in the epoch have been processed.

### Training arrays

- `X_train` contains the preprocessed training features.
- `y_train` contains the corresponding binary targets.
- Both must contain the same number of observations.

### Batch size

`batch_size=32` means that up to 32 observations contribute to one gradient update. A smaller batch uses less memory and produces noisier gradients; a larger batch usually uses more memory and produces smoother estimates. Thirty-two is a common starting value, not a universal optimum.

For $N$ training examples, the model performs approximately `ceil(N / 32)` parameter updates per epoch. Keras normally shuffles array-based training data between epochs unless configured otherwise.

### Epochs

One **epoch** is one complete pass through the training set. `epochs=100` sets a maximum training duration; it does not guarantee that 100 epochs are necessary or optimal. Too few epochs can underfit, while too many can overfit.

Do not choose the epoch count solely from training accuracy. Monitor held-out validation loss and stop when generalization stops improving.

## 4. Monitor validation performance

A stronger training call reserves part of the training data for validation and restores the best observed weights:

```python
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
)

history = ann.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1,
)
```

Use validation data for model and hyperparameter decisions. Keep the test set untouched until the final evaluation. For ordered or time-dependent data, do not use a random validation split without considering chronology.

## 5. Read the training history

`fit()` returns a `History` object containing the recorded values for each epoch:

```python
history.history.keys()
# Typically: loss, accuracy, val_loss, val_accuracy
```

Interpret common patterns as follows:

- Training and validation loss both fall: learning is progressing.
- Training loss falls while validation loss rises: likely overfitting.
- Both losses remain high or flat: possible underfitting, poor optimization, or a data issue.
- Large fluctuations: the learning rate may be too high, batches may be very noisy, or preprocessing may need investigation.

A reported training accuracy near 0.86 means approximately 86% of the **training examples** were classified correctly under the metric's threshold. It does not by itself establish 86% accuracy on unseen customers; test-set evaluation is still required.

## 6. Reproducibility

Neural-network results can vary because of random weight initialization, data shuffling, and nondeterministic operations. Set seeds when repeatability matters:

```python
tf.keras.utils.set_random_seed(42)
```

A seed improves reproducibility but does not make every hardware and software configuration perfectly deterministic. Record package versions, preprocessing artifacts, hyperparameters, and the data split alongside the model.

## 7. Common pitfalls

- Misspelling `Adam` as "Atom" or `epochs` as "epics."
- Treating accuracy as the loss function or assuming it is always sufficient.
- Pairing a sigmoid output with a loss configured for raw logits.
- Using categorical cross-entropy without checking whether labels are integers or one-hot vectors.
- Assuming a batch size of 32 or 100 epochs is universally optimal.
- Selecting hyperparameters from final test-set performance.
- Mistaking high training accuracy for good generalization.
- Ignoring validation loss while training for a fixed number of epochs.
- Forgetting to preserve exactly the same fitted preprocessing pipeline for inference.

## 8. Key takeaways

- `compile()` selects the learning objective, optimizer, and reported metrics.
- Adam updates parameters; binary cross-entropy measures the binary prediction error.
- `fit()` trains the model batch by batch over one or more epochs.
- Batch size and epoch count are hyperparameters, not fixed rules.
- Validation performance is the main guide during model development.
- Training accuracy must not be presented as test accuracy.
- Match the output activation, label encoding, and loss function.

## 9. Review questions

1. What is the difference between `compile()` and `fit()`?
2. What role does the optimizer play during training?
3. Why does this model use binary cross-entropy?
4. When should `from_logits=True` be used?
5. Why is `metrics` passed as a list?
6. What happens during one gradient update?
7. What is the difference between a batch and an epoch?
8. How does batch size affect optimization and memory use?
9. Why might 100 epochs be too many?
10. What does rising validation loss alongside falling training loss suggest?
11. Why is 86% training accuracy not enough to judge the model?
12. Which loss should be used for integer-coded multiclass labels?

## Further reading

- [TensorFlow: training and evaluation with Keras](https://www.tensorflow.org/guide/keras/training_with_built_in_methods)
- [TensorFlow API: `Model.compile`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile)
- [TensorFlow API: `Model.fit`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit)
- [TensorFlow API: Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)
- [TensorFlow API: binary cross-entropy](https://www.tensorflow.org/api_docs/python/tf/keras/losses/BinaryCrossentropy)
- [TensorFlow: overfitting and underfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit)

# Study Notes: Making Predictions and Evaluating the ANN

## 1. Objectives

This section uses the trained network to:

1. Predict the churn risk for one new customer.
2. Convert sigmoid scores into binary class predictions.
3. Generate predictions for the test set.
4. Evaluate the model with a confusion matrix and classification metrics.

The central rule is that inference data must undergo **exactly the same fitted preprocessing** as the training data, with the same feature meanings, order, encoding, and scaling.

## 2. Predict one customer's churn risk

The homework customer has these original values:

| Feature | Value |
|---|---:|
| Geography | France |
| Credit score | 600 |
| Gender | Male |
| Age | 40 |
| Tenure | 3 |
| Balance | 60,000 |
| Number of products | 2 |
| Has credit card | Yes |
| Active member | Yes |
| Estimated salary | 50,000 |

After applying the notebook's encodings, the model-ready feature order is:

```text
France, Germany, Spain, CreditScore, Gender, Age, Tenure,
Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary
```

For this fitted encoder, France is represented by `[1, 0, 0]` and male by `1`, giving the unscaled vector:

```python
customer = np.array([[
    1, 0, 0,      # Geography: France
    600,          # CreditScore
    1,            # Gender: Male
    40,           # Age
    3,            # Tenure
    60000,        # Balance
    2,            # NumOfProducts
    1,            # HasCrCard
    1,            # IsActiveMember
    50000,        # EstimatedSalary
]])
```

The outer brackets represent the batch; the inner brackets represent one observation. The resulting shape is `(1, 12)`. Keras models generally receive a batch dimension even when predicting only one observation.

## 3. Reuse the fitted scaler

The ANN was trained on standardized features, so the new customer must be transformed with the same fitted `StandardScaler`:

```python
customer_scaled = sc.transform(customer)
```

Use `transform()`, not `fit_transform()`. Refitting on a new customer would replace the training-set means and standard deviations, make the representation inconsistent with training, and invalidate the prediction. A single-row refit would also collapse standardized numeric values toward zero.

The same principle applies to every learned preprocessing step: never relearn categories, imputation values, scaling statistics, or column order during inference.

## 4. Obtain a score and a class prediction

Because the output layer contains one sigmoid unit, `predict()` returns a score between 0 and 1 for each observation:

```python
churn_score = ann.predict(customer_scaled, verbose=0).item()
churn_prediction = int(churn_score > 0.5)

print(f'Churn score: {churn_score:.3f}')
print(f'Predicted class: {churn_prediction}')
```

The notebook's compact equivalent is:

```python
print(ann.predict(sc.transform([[
    1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000
]]), verbose=0) > 0.5)
```

A Boolean result of `False` corresponds to class `0`—the model predicts that the customer will stay. `True` corresponds to class `1`—the model predicts churn.

A lecture run may produce a score near 0.04, but that exact value is not guaranteed. Random initialization, data shuffling, TensorFlow versions, and hardware can lead to different trained weights and scores. Interpret the result produced by the current trained model.

## 5. Scores are not decisions

The condition `score > 0.5` is a decision rule layered on top of the model score. The threshold controls the tradeoff between false positives and false negatives:

- **Lower threshold:** predicts more customers as churners, generally increasing recall but producing more false alarms.
- **Higher threshold:** predicts fewer customers as churners, generally increasing precision but missing more genuine churners.

A threshold of 0.5 is a conventional starting point, not automatically the best business threshold. Select it on validation data according to intervention costs and objectives—not by inspecting the final test set.

For example, if contacting a customer is inexpensive but missing a churner is costly, lowering the threshold may be appropriate. If retention offers are expensive, the bank may instead favor higher precision.

The sigmoid output is a model score trained with a probabilistic loss. It should not automatically be assumed to be perfectly calibrated; assess calibration when the numeric probability itself drives decisions.

## 6. Avoid brittle manual preprocessing

The vector `[1, 0, 0, ...]` works only because it matches this notebook's fitted category order and column order. Hard-coded inference vectors are easy to break when preprocessing changes. Verify generated categories rather than relying on memory:

```python
ct.named_transformers_['encoder'].categories_
```

For production work, prefer a named input record and one saved preprocessing pipeline that performs encoding and scaling. This reduces the risk of swapped columns, mismatched categories, and training-serving skew. The preprocessor and ANN should be versioned and deployed together.

## 7. Predict the complete test set

`X_test` was already transformed with the training-fitted scaler, so it can be passed directly to the network:

```python
y_score = ann.predict(X_test, verbose=0).ravel()
y_pred = (y_score > 0.5).astype(int)
```

Keeping `y_score` and `y_pred` separate is clearer than overwriting the scores. The continuous scores are needed later for threshold analysis, ROC-AUC, PR-AUC, ranking, and calibration.

To inspect predicted and actual labels side by side:

```python
comparison = np.column_stack((y_pred, y_test))
print(comparison)
```

Visual inspection is useful for debugging, but it is not a reliable overall evaluation. Use aggregate metrics calculated across the full test set.

## 8. Evaluate with a confusion matrix

```python
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Exited']))
```

For binary labels ordered as `[0, 1]`, scikit-learn arranges the matrix as:

```text
[[TN, FP],
 [FN, TP]]
```

For the lecture's example matrix:

```text
[[1520,  75],
 [ 202, 203]]
```

the cells mean:

- **1,520 true negatives:** stayed and were predicted to stay.
- **75 false positives:** stayed but were predicted to leave.
- **202 false negatives:** left but were predicted to stay.
- **203 true positives:** left and were predicted to leave.

Always state which class is considered positive. Here, `Exited = 1` is the positive churn class.

## 9. Accuracy is only part of the result

For the example matrix, test accuracy is:

```text
accuracy = (TN + TP) / total
         = (1520 + 203) / 2000
         = 86.15%
```

That sounds strong, but the class-specific results expose an important limitation:

- Churn precision is approximately `203 / (203 + 75) = 73.0%`.
- Churn recall is approximately `203 / (203 + 202) = 50.1%`.
- Specificity for customers who stay is approximately `1520 / (1520 + 75) = 95.3%`.
- Balanced accuracy is approximately `(50.1% + 95.3%) / 2 = 72.7%`.

The model correctly classifies most customers overall, partly because most customers stay. At the 0.5 threshold, however, it detects only about half of the customers who actually churn. Whether this is acceptable depends on the bank's objective and the costs of the two error types.

A useful baseline is the accuracy from always predicting the majority class. The ANN should be compared with that baseline and with simpler models—not judged by an isolated percentage.

## 10. Evaluate scores as well as thresholded labels

Threshold-independent ranking metrics provide additional information:

```python
from sklearn.metrics import average_precision_score, roc_auc_score

print(f'ROC-AUC: {roc_auc_score(y_test, y_score):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_score):.4f}')
```

- ROC-AUC measures ranking performance across thresholds.
- PR-AUC focuses on precision-recall performance and is often more informative when the positive class is uncommon.
- Neither metric chooses the operational threshold; threshold selection still requires validation data and business costs.

Keras can also report the compiled test loss and metrics directly:

```python
test_loss, test_accuracy = ann.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
```

Use `evaluate()` for the model's compiled metrics and the continuous scores with scikit-learn when more detailed diagnostic metrics are required.

## 11. Common pitfalls

- Passing a one-dimensional feature vector instead of a batch-shaped input.
- Supplying raw strings or unscaled numbers to a model trained on encoded, scaled features.
- Calling `fit_transform()` on a customer or the test set.
- Assuming that France will always map to `[1, 0, 0]` without checking the fitted encoder.
- Hard-coding positional feature vectors that silently break when preprocessing changes.
- Expecting the exact same prediction score from every training run.
- Overwriting probability scores with Boolean labels and losing ranking information.
- Treating 0.5 as the universally optimal threshold.
- Reversing false positives and false negatives when interpreting the confusion matrix.
- Reporting accuracy without examining churn recall, precision, and class balance.
- Tuning the threshold or model repeatedly on the final test set.

## 12. Key takeaways

- A single observation must include a batch dimension and the expected number of features.
- Reuse every preprocessing object fitted on the training data.
- A sigmoid output produces a score; a threshold converts that score into a decision.
- Lower and higher thresholds trade recall against false positives and precision.
- Preserve both continuous scores and thresholded labels.
- Read a binary confusion matrix as `[[TN, FP], [FN, TP]]`.
- An 86% accuracy can coexist with weak churn recall.
- Choose model settings and thresholds with validation data, then evaluate once on the held-out test set.

## 13. Review questions

1. Why does one customer still need a two-dimensional input?
2. Why must `sc.transform()` be used instead of `sc.fit_transform()`?
3. What does the sigmoid output represent?
4. Why might two training runs produce different scores for the same customer?
5. What happens to recall when the classification threshold is lowered?
6. Why should probability scores be retained separately from class labels?
7. How are the four entries of a binary confusion matrix arranged?
8. Which error represents a real churner predicted to stay?
9. Why can accuracy be misleading for churn prediction?
10. What does approximately 50% churn recall mean operationally?
11. Why should the decision threshold be selected on validation rather than test data?
12. How can a preprocessing pipeline make inference safer?

## Further reading

- [TensorFlow API: `Model.predict`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#predict)
- [TensorFlow API: `Model.evaluate`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate)
- [TensorFlow: classification on imbalanced data](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data)
- [scikit-learn: confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [scikit-learn: classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [scikit-learn: probability calibration](https://scikit-learn.org/stable/modules/calibration.html)